# 🚗 Dehazing and YOLOv8


---

## 🎯 **Objective**
To build a lightweight, real-time system that:
- Enhances foggy dashcam footage using **CLAHE-based dehazing**
- Detects objects like **cars, buses, trucks, potholes** using **YOLOv8**
- Compares detection accuracy between original and dehazed frames
- Aims to improve safety in autonomous and surveillance systems under poor visibility

---

## 🔧 **Pipeline Overview**

1. **Video Input** — Load real-world foggy dashcam footage
2. **Dehazing with CLAHE** — Enhance image contrast to improve visibility
3. **Object Detection** — Use YOLOv8 for detecting target classes
4. **Side-by-Side Comparison** — Show detections on both original & dehazed frames
5. **Detection Count Analysis** — Count & compare detections frame-by-frame

---

## 📊 Project Pipeline




---

## 🔧 Techniques Used

- **CLAHE (Contrast Limited Adaptive Histogram Equalization)** for dehazing
- **YOLOv8** pre-trained model (via Ultralytics) for object detection
- **OpenCV + Matplotlib** for video frame handling and visualization

---

## ✅ Evaluation Metric

- **Average Detections per Frame** on both:
  - Original video
  - Dehazed video

## Import Libraries

In [1]:
# Install YOLOv8 package (only once in Colab/Jupyter)
# !pip install ultralytics

# Import required modules
import cv2
import numpy as np
from ultralytics import YOLO
from IPython.display import display, clear_output
import matplotlib.pyplot as plt


📘 This cell sets up the core libraries needed for image processing, plotting, and YOLO inference.

## Load Model & Target Labels

In [2]:
# Load pre-trained YOLOv8 nano model
model = YOLO('yolov8n.pt')

# Only detect these relevant road objects
TARGET_CLASSES = ['car', 'bus', 'truck', 'pothole']

📘 Loads YOLOv8 and selects classes we want to focus on detecting (others ignored).

## Dehazing with CLAHE

In [3]:
def clahe_dehaze(image):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    l = clahe.apply(l)

    lab = cv2.merge((l, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


📘 Improves frame contrast using histogram equalization in LAB color space, simulating fog removal.

## Object Detection and Visualization

In [4]:
def detect_and_draw(image, label_filter):
    results = model(image, verbose=False)
    detections = 0
    
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            label = result.names[int(box.cls[0])]
            conf = box.conf[0].item()

            if label in label_filter:
                detections += 1
                cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(image, f"{label} {conf:.2f}", (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    return image, detections


## Main Pipeline Function

In [5]:
def main(video_path, max_frames=20):
    cap = cv2.VideoCapture(video_path)
    frame_index = 0
    total_raw_detections = 0
    total_dehazed_detections = 0

    while cap.isOpened() and frame_index < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        dehazed = clahe_dehaze(frame.copy())

        frame_raw, count_raw = detect_and_draw(frame.copy(), TARGET_CLASSES)
        frame_dehazed, count_dehaze = detect_and_draw(dehazed.copy(), TARGET_CLASSES)

        total_raw_detections += count_raw
        total_dehazed_detections += count_dehaze

        # Convert to RGB for visualization
        raw_rgb = cv2.cvtColor(frame_raw, cv2.COLOR_BGR2RGB)
        dehazed_rgb = cv2.cvtColor(frame_dehazed, cv2.COLOR_BGR2RGB)

        # Show side-by-side
        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.imshow(raw_rgb)
        plt.title(f"Original | Detections: {count_raw}")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(dehazed_rgb)
        plt.title(f"Dehazed | Detections: {count_dehaze}")
        plt.axis('off')

        plt.tight_layout()
        display(plt.gcf())
        plt.close()

        frame_index += 1
        clear_output(wait=True)

    cap.release()

    # Calculate average detections per frame
    accuracy_raw = total_raw_detections / max_frames if max_frames else 0
    accuracy_dehazed = total_dehazed_detections / max_frames if max_frames else 0

    return accuracy_raw, accuracy_dehazed


 Handles the entire video: reads frames, applies dehazing + detection, and shows side-by-side comparisons.

## Run and Show Accuracy

In [6]:
# Run the video through the pipeline
accuracy_orig, accuracy_dehazed = main("P2.mp4", max_frames=300)

# Print average detections per frame
print(f"📊 Original Video - Avg Detections/frame: {accuracy_orig:.2f}")
print(f"📊 Dehazed Video  - Avg Detections/frame: {accuracy_dehazed:.2f}")


📊 Original Video - Avg Detections/frame: 1.02
📊 Dehazed Video  - Avg Detections/frame: 1.20


## 📈 Outcome

We measure and compare the total number of valid detections across both versions to determine if **dehazing improves YOLOv8 detection accuracy**.

---


## 📊 **Comparison with Existing Models**

| Feature | 🟢 **Our Model** | 🔵 AOD-Net + YOLO | 🟣 Joint Dehaze + Detect Models |
|--------|------------------|------------------|-------------------------------|
| **Architecture** | Simple pipeline | 2-stage pipeline | End-to-end CNN |
| **Dehazing** | CLAHE (traditional) | AOD-Net (CNN) | Learned dehazing layers |
| **Detection** | YOLOv8 pretrained | YOLOv5/YOLOv8 | Custom detection head |
| **Real-time Ready?** | ✅ Yes | ⚠️ Medium | ❌ Slow |
| **Training Needed?** | ❌ None | ✅ Yes (AOD-Net) | ✅ Yes |
| **Data** | Real foggy video | Mostly synthetic | Mostly synthetic |
| **Comparison Output** | ✅ Shown | ❌ Not built-in | ❌ Not visualized |
| **Deployability** | ✅ Easy | ⚠️ Moderate | ❌ Complex |

---

## 💎 **What Makes This Project Unique**

- 🔋 **Lightweight and Efficient**: Uses CLAHE instead of deep models — real-time friendly.
- 🎥 **Real-world Video Compatible**: Works directly on foggy dashcam footage.
- 📈 **Clear Accuracy Comparison**: Displays detection count frame-by-frame.
- 🔌 **Easy to Integrate**: Works with OpenCV and YOLOv8 — minimal setup.
- 🔁 **Modular Pipeline**: Can replace CLAHE with any advanced dehazing model easily.

---

## 🚀 **Future Goals to Improve Accuracy**

1. 🔄 Replace CLAHE with AOD-Net or a deep dehazing CNN for better restoration.
2. 🧠 Train YOLOv8 on synthetic + real foggy datasets (e.g., Foggy Cityscapes).
3. 📐 Add evaluation metrics like mAP, IoU, precision, recall.
4. 🧪 Experiment with transformer-based object detectors.
5. 💥 Deploy on embedded systems like NVIDIA Jetson Nano for real-world testing.




